# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [52]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [4]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [5]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be dealing with or mismanaging the lender or servicer. Specific problems include errors in loan balances, misapplied payments, wrongful denials of payment plans, inaccurate or disputed information on credit reports, problems with loan transfers and communication, and issues related to loan forgiveness or discharge. Many complaints also involve borrowers being misled about their repayment obligations, interest capitalization, or being placed into long-term forbearance without proper notice.\n\nIn summary, a prevalent and overarching issue is the mismanagement or mishandling of student loans by lenders or servicers, leading to errors, confusion, and hardship for borrowers.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints were not handled in a timely manner. Specifically, the complaint filed by the consumer with MOHELA on 03/28/25 was marked as "Not timely," indicating it was not responded to within the expected timeframe. Additionally, multiple complaints mention ongoing issues and delays in resolution, such as the complaint with Maximus Federal Services, Inc. from 04/14/25, which still had not been resolved after over a year, and others with similar unresolved or prolonged issues.'

In [13]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to a combination of factors highlighted in the complaints:\n\n1. **Lack of clear communication and notification:** Many borrowers were not properly informed about when their repayment was to resume, changes in loan servicers, or the transfer of their loans. This led to missed payments and misunderstandings about the status of their debts.\n\n2. **Compounding interest and repayment challenges:** Borrowers expressed that options such as forbearance or deferment result in interest continuing to accumulate, making the total debt larger over time and prolonging repayment periods.\n\n3. **Financial hardships and unrealistic repayment expectations:** Economic difficulties, stagnant wages, and high living costs made consistent repayment difficult. Some borrowers felt misled about the affordability and terms of repayment plans.\n\n4. **Technological and administrative issues:** Errors in applying payments, reporting late payments, or transfer

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, specifically issues such as disagreements over fees, difficulty applying payments correctly, getting inaccurate or bad information about loans, and lack of transparency. Many complaints involve the servicers' handling of payments, loan details, and the trustworthiness of the information provided.\n\nSo, the most common issue with loans, according to this data, is difficulties and disputes arising from interactions with lenders or loan servicers, particularly related to miscommunication, incorrect application of payments, and incorrect or confusing information."

In [17]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, all of the complaints indicate that the companies responded in a timely manner. For example, the complaints received responses marked as "Timely response?": "Yes" and were closed with explanations. There is no indication of any complaints not being handled promptly.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including miscommunication, errors, and issues with the loan servicing process. Some specific causes mentioned in the complaints are:\n\n- Being steered into improper forbearances or payment plans that increased the principal.\n- Loan transfers to new servicers like Aidvantage without proper notification, leading to unenrollment from autopay and lack of awareness about account status.\n- Poor communication from loan servicers, such as not informing borrowers about account changes, overdue payments, or the need to take action.\n- Technical problems like reversed payments, errors with bank information, or failure to respond to requests for deferment or forbearance, resulting in missed payments.\n- Lack of timely responses or assistance from the loan servicing companies when issues arose.\n- Receipt of bad or misleading information about the status of their loans.\n\nOverall, many borrowers experienced failures due to inadequate 

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

✅ Suppose for a query like "P0420 catalytic converter diagnostic code", BM25 excels with this query due to the highly specific, rare, and non-semantic terms ("P0420"). Its term-frequency and inverse document frequency (IDF) components will prioritize documents containing these exact, crucial identifiers, which are vital for exact matches in technical domains. Embeddings might generalize to "engine problems" and miss the precise code.




## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans involves problems related to the handling and management of the loans, such as errors in loan balances, misapplied payments, wrong or bad information, and mishandling of loan data. Specifically, there are recurring concerns about inaccuracies in loan amounts, lack of clear communication, improper transfers of loans, and violations of privacy and rights.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. For example, the complaint from the individual regarding their student loans and the lack of response for over a year indicates delays in addressing their concern. Additionally, the complaint from the other individual about payments not appearing on their account has been ongoing for over 2-3 weeks without resolution.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of understanding and information: Borrowers were often unaware that they needed to repay their loans or were not adequately informed about the repayment process, interest accumulation, and loan terms.\n\n2. Administrative issues and communication breakdowns: Borrowers experienced difficulties due to poor communication from loan servicers, such as not being notified about payment obligations, loan transfers without their knowledge, or being locked out of online accounts with incorrect information.\n\n3. Accumulation of interest and unpaid balances: When loans were placed into forbearance or deferment, interest continued to accrue, leading to increased balances over time. Lower monthly payments extended the repayment period and increased total interest paid.\n\n4. Insufficient or misleading options for repayment: Borrowers were often only offered options like forbearance or deferment, which could worsen the

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [24]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [25]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be dealing with the loan servicer or lender, including problems such as mismanagement, errors in loan balances, misapplied payments, wrongful denials of payment plans, lack of proper communication or notices, and inaccuracies in reporting to credit bureaus. Many complaints highlight challenges in obtaining accurate information, corrections, or assistance from the loan servicers, which significantly impacts borrowers’ financial stability and credit standing.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints indicate that complaints were not handled in a timely manner. Specifically:\n\n- Several complaints explicitly mention that responses were delayed, such as complaint ID 12709087 regarding a federal student loan application, which was marked as "No" for timely response.\n- Multiple complaints from EdFinancial Services (e.g., IDs 12823876, 13070146, and 13056764) state that the company responded "with explanation" but emphasize ongoing issues, delays, or unresolved concerns despite follow-ups, indicating delayed or insufficient handling.\n- A complaint (ID 12739706) about a delay exceeding 30 days in an investigation indicates a prolonged unresolved situation.\n- Several complaints against Maximus/AidyVantage mention that responses failed or that the issue persisted for extended periods (e.g., over a year or multiple months), sometimes with no response at all.\n\nIn summary, yes, there are complaints that suggest some issues were no

In [28]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily because they faced financial hardships, lack of proper information, and difficulties managing their loans. Specific reasons include:\n\n- Accumulation of interest during forbearance or deferment periods, which made loans harder to pay off and increased total debt.\n- Lack of clear communication and transparency from loan servicers about repayment options like income-driven repayment plans, loan forgiveness, or the risks of certain payment strategies.\n- Being steered into long-term forbearance or consolidation without being informed of alternative solutions that might preserve benefits or reduce interest.\n- Unexpected or unnotified transfer of loans between servicers, leading to missed payments and credit reporting errors.\n- Mismanagement or inaccuracies in loan balances and account information, sometimes coupled with illegal disclosures or violations of privacy laws like FERPA.\n- Insufficient assistance or guidance from servicers whe

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.


Generating multiple reformulations of a user query can imporve recall by: 

- **Overcomes Vocabulary Mismatch**: Finds documents using synonyms or different phrasing.

- **Adjusts Specificity**: Relaxes overly narrow queries or expands general ones.

- **Captures User Intent**: Better understands the user's underlying information need.

- **Broadens Search Scope**: Increases the chance of hitting all relevant documents.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be related to errors and mismanagement by loan servicers. Specific recurring problems include incorrect or misleading information on credit reports, errors in loan balances, misapplied payments, wrongful denials of payment plans, and disputes over interest rates and fees. People also report issues with the handling of their accounts after loans are sold or transferred, leading to confusion and unverified or questionable debts being reported or collected.\n\nIn summary, the most common issue is **mismanagement and inaccuracies in loan servicing and reporting**, which can cause significant financial and emotional hardship for borrowers.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, several complaints were marked as "No" for a timely response, indicating they did not get handled in a timely manner. Specifically, the complaints with Complaint IDs 12709087 and 12935889 both state "Timely response?": "No". This suggests that these complaints were not handled promptly by the companies.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors including confusion or lack of proper communication from loan servicers, and severe financial hardships resulting from unforeseen circumstances or misrepresentations. For example, some borrowers experienced issues with improper billing or lack of clear information about payment start dates and grace periods, which led them to miss payments. Others faced significant financial difficulties due to job loss, health issues, or the long-term consequences of studying at institutions that misrepresented their value or faced financial instability, making it difficult to sustain loan payments. In some cases, borrowers also struggled with complexities related to loan management and the accuracy of debt reporting, which further complicated repayment efforts.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be dealing with the loan servicers and lenders regarding inaccurate or bad information about the loans, including errors in loan balances, interest calculations, and wrongful denials of repayment plans. Many complaints also highlight issues like lack of transparency, improper handling of loan transfers, and difficulty obtaining accurate information or assistance.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, yes, some complaints indicate that complaints did not get handled in a timely manner. Specifically:\n\n- Complaint ID 12935889 (submitted 04/11/25, Maximus Federal Services, Inc. in CO) was marked as **"No"** under "Timely response?" which suggests it was not handled within the expected timeframe.\n- Similarly, Complaint ID 12709087 (submitted 03/28/25, MOHELA in CA) was marked as **"No"** under "Timely response?" indicating a delay.\n\nMost other complaints show "Yes" for timely response, implying they were handled within the expected period. \n\nIn summary, at least some complaints were not handled in a timely manner, as evidenced by the complaints that received a "No" response to the timeliness query.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for several reasons based on the complaints:\n\n1. **Lack of Clear Communication and Notifications:** Many borrowers were not adequately informed about when their payments were due, the status of their loans, or changes in servicing. Some were unaware their loans had been transferred to new servicers or that repayment had resumed, leading to unintentional delinquencies.\n\n2. **Complex and Confusing Payment Processes:** Borrowers reported difficulty applying payments correctly, ensuring that extra funds went toward principal rather than interest, and understanding their loan balances and interest accrual. This confusion sometimes resulted in missed or late payments.\n\n3. **Inability to Afford Payments:** Many individuals face financial hardship and were unable to increase payments without sacrificing basic necessities. Increasing payments was often not feasible as it would extend repayment periods and increase total interest, making repayment see

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [44]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"The most common issue with loans, based on the provided complaints, appears to be related to problems with loan servicing and communication. This includes issues such as:\n\n- Struggling to repay or problems with forgiveness, cancellation, or discharge.\n- Errors or improper use of credit reports and reporting disputes.\n- Lack of transparency and accountability from loan servicers.\n- Problems with payment processing, auto-debits, or changes in payment terms.\n- Confusion or inconsistency about the loan issuer and account status.\n- Unauthorized access or breaches related to privacy and data security.\n- Failure to verify or process applications correctly, leading to incorrect billing or account status.\n\nOverall, a significant number of complaints highlight frustrations with loan servicers' handling of accounts, inadequate communication, and failure to resolve issues promptly."

In [48]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, several complaints indicate that there were delays or issues in handling responses or resolving issues in a timely manner. Specifically, there are complaints where the companies responded "Closed with explanation" or similar, which suggests that the complaints were addressed, but some issues remained ongoing or unresolved from the complainants\' perspective.\n\nHowever, the provided data does not explicitly state any complaints that definitively were *not* handled in a timely manner. Most responses indicate "Yes" under the "Timely response?" field, meaning the companies responded within the expected or indicated timeframes.\n\nTherefore, I do not have conclusive evidence from this data that any complaints did not get handled in a timely manner.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including difficulties dealing with lenders or servicers, problems with loan documentation or forgiveness processes, issues arising from erroneous or incomplete information, and technical or administrative errors by the loan servicers. Some borrowers also faced challenges due to delays or failures in re-amortizing payments after periods of forbearance, or because of disputes over the legitimacy or legality of their loans, such as illegal reporting or data breaches. Additionally, issues like miscommunication, lack of transparency, or delays in resolving account statuses contributed to borrowers' inability to repay their loans effectively."

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

✅ If sentences are short and highly repeitive, Semantic chunking sees each short sentence as its own chunk, even if they're part of one aswer. This leads to context missing the bigger picture making it harder for algorithm to get full meaning leading to less accurate grouping.

To get around this behaviour:
- Group similar sentences as one long "sentence"
- Adjust the similarity to get fewer, bigger chunks.
- Combine tiny chunks if they're related

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [54]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from ragas.testset import TestsetGenerator


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

dataset.to_pandas()


Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '388d76'. Skipping!
Property 'summary' already exists in node 'a2cbf9'. Skipping!
Property 'summary' already exists in node 'be5b26'. Skipping!
Property 'summary' already exists in node 'c9ba6b'. Skipping!
Property 'summary' already exists in node 'f1563a'. Skipping!
Property 'summary' already exists in node '5f86ac'. Skipping!
Property 'summary' already exists in node 'd6d2cf'. Skipping!
Property 'summary' already exists in node '3fe2e4'. Skipping!
Property 'summary' already exists in node '184c59'. Skipping!
Property 'summary' already exists in node '81efd5'. Skipping!
Property 'summary' already exists in node '13414d'. Skipping!
Property 'summary' already exists in node '73ab64'. Skipping!
Property 'summary' already exists in node '85ab08'. Skipping!
Property 'summary' already exists in node '3acc13'. Skipping!


...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Where can I find more detailed information abo...,"[non-term (includes clock-hour calendars), or ...",For more detail on subscription-based programs...,single_hop_specifc_query_synthesizer
1,Whaat are the requirments for includin clinnic...,[Inclusion of Clinical Work in a Standard Term...,Clinnical work in standerd term programs in me...,single_hop_specifc_query_synthesizer
2,"What Non-Term Characteristics is, like, when p...",[Non-Term Characteristics A program that measu...,Non-Term Characteristics mean a program is con...,single_hop_specifc_query_synthesizer
3,Where can I find guidance on calculating Pell ...,[both the credit or clock hours and the weeks ...,For guidance on calculating Pell Grant and TEA...,single_hop_specifc_query_synthesizer
4,what is standard term and what is standard ter...,[<1-hop>\n\nnon-term (includes clock-hour cale...,"A standard term is a period like a semester, t...",multi_hop_abstract_query_synthesizer
5,if a student in a medical program has to do pr...,[<1-hop>\n\nInclusion of Clinical Work in a St...,If the practicum or clinical experience is req...,multi_hop_abstract_query_synthesizer
6,How does the structure of a subscription-based...,[<1-hop>\n\nnon-term (includes clock-hour cale...,"In a subscription-based academic calendar, whi...",multi_hop_abstract_query_synthesizer
7,How does the structure of a subscription-based...,[<1-hop>\n\nnon-term (includes clock-hour cale...,"In a subscription-based academic calendar, whi...",multi_hop_abstract_query_synthesizer
8,"How Volume 8, Chapter 3 talk about disbursemen...",[<1-hop>\n\nDisbursement Timing in Subscriptio...,"Volume 8, Chapter 3 provides guidance on disbu...",multi_hop_specific_query_synthesizer
9,when student in nonstandard term finish more h...,[<1-hop>\n\nInclusion of Clinical Work in a St...,if student in nonstandard term program finish ...,multi_hop_specific_query_synthesizer


In [68]:
import copy
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import RunConfig
import time


def make_chat_model(retriever_name: str = None, model_name: str = "gpt-4o") -> ChatOpenAI:
    project_name=f"ragas-rkv-eval-{retriever_name}"
    print(project_name)
    if retriever_name:
        tracer = LangChainTracer(
            project_name=project_name,
            tags=[retriever_name, "retriever", model_name]
        )
        callback_manager = CallbackManager([tracer])
        return ChatOpenAI(
            model=model_name,
            temperature=0,
            max_tokens=8192,
            callback_manager=callback_manager
        )

retrievers = {
    "naive_retriever": naive_retrieval_chain,
    "bm25_retriever": bm25_retrieval_chain,
    "compression_retriever": contextual_compression_retrieval_chain,
    "multi_query_retriever": multi_query_retrieval_chain,
    "parent_document_retriever": parent_document_retrieval_chain,
    "ensemble_retriever": ensemble_retrieval_chain
}


limited_dataset = list(dataset)[:1]  # Only first 10 cases

results = {}
for retriever_name in retrievers:
    _retriever = retrievers[retriever_name]
    for dataset_row in limited_dataset:
        # invoke the retriever with the generated user input
        retriever_response = _retriever.invoke({"question": dataset_row.eval_sample.user_input})
        # evaluation
        dataset_row.eval_sample.response = retriever_response["response"].content
        dataset_row.eval_sample.retrieved_contexts = [context.page_content for context in retriever_response["context"]]

    evaluator_llm = LangchainLLMWrapper(make_chat_model(retriever_name=retriever_name,model_name="gpt-4.1-mini"))

    evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas().head(1))
    custom_run_config = RunConfig(timeout=360)

    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
        llm=evaluator_llm,
        run_config=custom_run_config
    )

    results[retriever_name] = result



ragas-rkv-eval-naive_retriever


/var/folders/19/mmb107hd3s54vcpxc0wqjp2h0000gn/T/ipykernel_83914/3593325370.py:19: DeprecationWarning: callback_manager is deprecated. Please use callbacks instead.
  return ChatOpenAI(


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

ragas-rkv-eval-bm25_retriever


/var/folders/19/mmb107hd3s54vcpxc0wqjp2h0000gn/T/ipykernel_83914/3593325370.py:19: DeprecationWarning: callback_manager is deprecated. Please use callbacks instead.
  return ChatOpenAI(


...

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

ragas-rkv-eval-ensemble_retriever


/var/folders/19/mmb107hd3s54vcpxc0wqjp2h0000gn/T/ipykernel_83914/3593325370.py:19: DeprecationWarning: callback_manager is deprecated. Please use callbacks instead.
  return ChatOpenAI(


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

Exception raised in Job[4]: LLMDidNotFinishException(The LLM generation was not completed. Please increase try increasing the max_tokens and try again.)
Exception raised in Job[5]: TimeoutError()


In [69]:
for key in results:
    print(f"{key}:{results[key]}")

naive_retriever:{'context_recall': 0.0000, 'faithfulness': 0.0000, 'factual_correctness': 0.2700, 'answer_relevancy': 0.0000, 'context_entity_recall': 0.0000, 'noise_sensitivity_relevant': 0.0000}
bm25_retriever:{'context_recall': 0.0000, 'faithfulness': 0.5714, 'factual_correctness': 0.4000, 'answer_relevancy': 0.0000, 'context_entity_recall': 0.0000, 'noise_sensitivity_relevant': 0.0000}
compression_retriever:{'context_recall': 0.0000, 'faithfulness': 0.3333, 'factual_correctness': 0.2500, 'answer_relevancy': 0.0000, 'context_entity_recall': 0.0000, 'noise_sensitivity_relevant': 0.0000}
multi_query_retriever:{'context_recall': 0.0000, 'faithfulness': 0.0000, 'factual_correctness': 0.6700, 'answer_relevancy': 0.0000, 'context_entity_recall': 0.0000, 'noise_sensitivity_relevant': 0.0000}
parent_document_retriever:{'context_recall': 0.0000, 'faithfulness': 0.2667, 'factual_correctness': 0.5000, 'answer_relevancy': 0.0000, 'context_entity_recall': 0.0000, 'noise_sensitivity_relevant': 0.

| Retriever           | Context Recall | Faithfulness | Factual Correctness | Cost     | Latency  | Notes                                             |
| ------------------- | -------------- | ------------ | ------------------- | -------- | -------- | ------------------------------------------------- |
| **Naive**           | 0.0000         | 0.0000       | 0.2700              | ✅ Low    | ✅ Fast   | Fails across all critical metrics                 |
| **BM25**            | 0.0000         | 0.5714       | 0.4000              | ✅ Low    | ✅ Fast   | High faithfulness; no context retrieved           |
| **Compression**     | 0.0000         | 0.3333       | 0.2500              | ❌ High   | ❌ Slow   | Poor context recall; not cost-effective           |
| **Multi-query**     | 0.0000         | 0.0000       | **0.6700**          | ❌ Medium | ❌ Medium | Best factual accuracy but not grounded in context |
| **Parent-document** | 0.0000         | 0.2667       | 0.5000              | ✅ Low    | ✅ Fast   | Performs decently; similar to BM25                |
| **Ensemble**        | **1.0000**     | 0.5000       | 0.3300              | ❌ High   | ❌ Slow   | Only method that retrieved correct context        |




Among all methods, BM25 offers the best trade-off between performance, cost, and latency. It has the highest faithfulness (0.57) and decent factual correctness (0.40), despite retrieving no exact ground-truth context.

Ensemble retriever is the only one with perfect context recall, but it comes with high cost and latency. It’s best suited for high-accuracy applications.

Multi-query retriever achieves the highest factual correctness (0.67) but lacks grounding, making it less reliable.